In [42]:
import pandas as pd
import random
from datetime import datetime, timedelta

In [43]:
NUM_RECORDS = 100

SCENARIOS = [
    "NORMAL",
    "PARTIAL_REFUND",
    "SETTLEMENT_DELAY",
    "MISSING_BANK_RECORD",
    "UNEXPLAINED_MISMATCH",
]

#### Simulate payment gateway fee+ GST

In [44]:
def generate_fee(amount):
    """
    Simulate payment gateway fee + GST.
    """

    fee = round(amount * 0.02, 2)
    tax = round(fee * 0.18, 2)

    return fee, tax

#### Generate a random payment date

In [45]:
def generate_date():
    """
    Generate a random payment date.
    """

    start_date = datetime(2026, 8, 1)

    random_days = random.randint(0, 30)

    return start_date + timedelta(days=random_days)


#### Main Data Generation

In [46]:
def generate_data():

    payments = []
    settlements = []
    bank_records = []
    refunds = []
    ground_truth = []

    for i in range(1, NUM_RECORDS + 1):

        payment_id = f"pay_{i:04d}"
        order_id = f"order_{i:04d}"
        settlement_id = f"set_{i:04d}"

        amount = random.choice([
            500,
            1000,
            1500,
            2000,
            5000,
            10000,
            15000,
            25000
        ])

        payment_date = generate_date()

        scenario = random.choice(SCENARIOS)

        fee, tax = generate_fee(amount)

        # -----------------------------
        # PAYMENT RECORD
        # -----------------------------

        payments.append({
            "payment_id": payment_id,
            "order_id": order_id,
            "gross_amount": amount,
            "payment_date": payment_date.strftime("%Y-%m-%d"),
            "status": "captured"
        })

        # -----------------------------
        # NORMAL SCENARIO
        # -----------------------------

        if scenario == "NORMAL":

            net_amount = round(amount - fee - tax, 2)

            settlement_date = payment_date + timedelta(
                days=random.randint(1, 3)
            )

            settlements.append({
                "settlement_id": settlement_id,
                "payment_id": payment_id,
                "gross_amount": amount,
                "fee": fee,
                "tax": tax,
                "net_amount": net_amount,
                "settlement_date": settlement_date.strftime("%Y-%m-%d")
            })

            bank_records.append({
                "bank_transaction_id": f"bank_{i:04d}",
                "reference": settlement_id,
                "amount": net_amount,
                "credit_date": (
                    settlement_date + timedelta(days=1)
                ).strftime("%Y-%m-%d"),
                "description": "RAZORPAY SETTLEMENT"
            })

        # -----------------------------
        # PARTIAL REFUND
        # -----------------------------

        elif scenario == "PARTIAL_REFUND":

            refund_amount = random.choice([
                amount * 0.10,
                amount * 0.20,
                amount * 0.25
            ])

            refund_amount = round(refund_amount, 2)

            remaining_amount = amount - refund_amount

            fee, tax = generate_fee(remaining_amount)

            net_amount = round(
                remaining_amount - fee - tax,
                2
            )

            settlement_date = payment_date + timedelta(
                days=random.randint(1, 3)
            )

            refunds.append({
                "refund_id": f"refund_{i:04d}",
                "payment_id": payment_id,
                "refund_amount": refund_amount,
                "refund_date": (
                    payment_date + timedelta(days=1)
                ).strftime("%Y-%m-%d")
            })

            settlements.append({
                "settlement_id": settlement_id,
                "payment_id": payment_id,
                "gross_amount": amount,
                "fee": fee,
                "tax": tax,
                "net_amount": net_amount,
                "settlement_date": settlement_date.strftime("%Y-%m-%d")
            })

            bank_records.append({
                "bank_transaction_id": f"bank_{i:04d}",
                "reference": settlement_id,
                "amount": net_amount,
                "credit_date": (
                    settlement_date + timedelta(days=1)
                ).strftime("%Y-%m-%d"),
                "description": "RAZORPAY SETTLEMENT"
            })

        # -----------------------------
        # SETTLEMENT DELAY
        # -----------------------------

        elif scenario == "SETTLEMENT_DELAY":

            net_amount = round(amount - fee - tax, 2)

            # Deliberately delayed settlement
            settlement_date = payment_date + timedelta(
                days=random.randint(7, 14)
            )

            settlements.append({
                "settlement_id": settlement_id,
                "payment_id": payment_id,
                "gross_amount": amount,
                "fee": fee,
                "tax": tax,
                "net_amount": net_amount,
                "settlement_date": settlement_date.strftime("%Y-%m-%d")
            })

            bank_records.append({
                "bank_transaction_id": f"bank_{i:04d}",
                "reference": settlement_id,
                "amount": net_amount,
                "credit_date": (
                    settlement_date + timedelta(days=1)
                ).strftime("%Y-%m-%d"),
                "description": "RAZORPAY SETTLEMENT"
            })

        # -----------------------------
        # MISSING BANK RECORD
        # -----------------------------

        elif scenario == "MISSING_BANK_RECORD":

            net_amount = round(amount - fee - tax, 2)

            settlement_date = payment_date + timedelta(
                days=random.randint(1, 3)
            )

            settlements.append({
                "settlement_id": settlement_id,
                "payment_id": payment_id,
                "gross_amount": amount,
                "fee": fee,
                "tax": tax,
                "net_amount": net_amount,
                "settlement_date": settlement_date.strftime("%Y-%m-%d")
            })

            # Intentionally no bank record

        # -----------------------------
        # UNEXPLAINED MISMATCH
        # -----------------------------

        elif scenario == "UNEXPLAINED_MISMATCH":

            net_amount = round(amount - fee - tax, 2)

            settlement_date = payment_date + timedelta(
                days=random.randint(1, 3)
            )

            settlements.append({
                "settlement_id": settlement_id,
                "payment_id": payment_id,
                "gross_amount": amount,
                "fee": fee,
                "tax": tax,
                "net_amount": net_amount,
                "settlement_date": settlement_date.strftime("%Y-%m-%d")
            })

            # Introduce unexplained discrepancy
            incorrect_bank_amount = net_amount - random.choice([
                100,
                250,
                500
            ])

            bank_records.append({
                "bank_transaction_id": f"bank_{i:04d}",
                "reference": settlement_id,
                "amount": incorrect_bank_amount,
                "credit_date": (
                    settlement_date + timedelta(days=1)
                ).strftime("%Y-%m-%d"),
                "description": "RAZORPAY SETTLEMENT"
            })

        # -----------------------------
        # GROUND TRUTH
        # -----------------------------

        ground_truth.append({
            "payment_id": payment_id,
            "true_scenario": scenario
        })

    # -----------------------------
    # CONVERT TO DATAFRAMES
    # -----------------------------

    payments_df = pd.DataFrame(payments)
    settlements_df = pd.DataFrame(settlements)
    bank_df = pd.DataFrame(bank_records)
    refunds_df = pd.DataFrame(refunds)
    ground_truth_df = pd.DataFrame(ground_truth)

    # -----------------------------
    # SAVE FILES
    # -----------------------------

    payments_df.to_csv(
        r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\payments.csv",
        index=False
    )

    settlements_df.to_csv(
        r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\settlements.csv",
        index=False
    )

    bank_df.to_csv(
        r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\bank_records.csv",
        index=False
    )

    refunds_df.to_csv(
        r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\refunds.csv",
        index=False
    )

    ground_truth_df.to_csv(
        r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\ground_truth.csv",
        index=False
    )

    print("\nData generation complete!\n")

    print(f"Payments: {len(payments_df)}")
    print(f"Settlements: {len(settlements_df)}")
    print(f"Bank records: {len(bank_df)}")
    print(f"Refunds: {len(refunds_df)}")
    print(f"Ground truth records: {len(ground_truth_df)}")


if __name__ == "__main__":
    generate_data()


Data generation complete!

Payments: 100
Settlements: 100
Bank records: 87
Refunds: 17
Ground truth records: 100
